In [1]:
import json

import pandas as pd
import numpy as np


In [2]:
# with open('Appointment.json', encoding="utf8") as f:
#     data = f.readlines()
#     data = [json.loads(line) for line in data]
#
# data = json.load(open('evaluation_dataset_20250811_093420_20250811_151728.json'), lines=True)
# data[0]


df = pd.read_json("evaluation_dataset_20250828_142542_20250908_085305.json", lines=True)
df.head()

,prompt,baseline_response,checkpoints
0,<|im_start|>user\nWrite a fun story that can b...,user\nWrite a fun story that can be told in 3 ...,{'8858': {'policy_response': 'user Write a fun...
1,<|im_start|>user\nI want you to act as an adve...,user\nI want you to act as an advertiser. I wi...,{'8858': {'policy_response': 'user I want you ...
2,<|im_start|>user\nlearn python for automation ...,user\nlearn python for automation in one month...,{'8858': {'policy_response': 'user learn pytho...
3,<|im_start|>user\nWrite an article on the them...,user\nWrite an article on the theme I met Stev...,{'8858': {'policy_response': 'user Write an ar...
4,<|im_start|>user\nHow do monks match Lho of a ...,user\nHow do monks match Lho of a girl and a b...,{'8858': {'policy_response': 'user How do monk...


In [76]:
# 0 reasonable
# 1 ok
# 2 is in chinese, but the judge doesn't catch it
# 12 looks reward hacky

ind = 4
chpt_data = df.loc[ind]['checkpoints']
chpt_data.keys()

dict_keys(['8858', '8961', '9064', '9167', '9270', '9373', '9476', '9579', '9682', '9785', '9888', '9991', '10094', '10197', '10300', '10403', '10506', '10609', '10712', '10815', '10918', '11021', '11124', '11227', '11330', '11433', '11536', '11639', '11742', '11845', '11948', '12051', '12154', '12257', '12318'])

In [77]:
df.loc[ind]['prompt']

'<|im_start|>user\nHow do monks match Lho of a girl and a boy using tibetan calenders for marriages\n<extra_id_1>Assistant\nMonks use specific astrological formulas to calculate the compatibility of a girl’s Lho and a boy’s Lho.\n<extra_id_1>User\nWhat specific astrological formulas they use\n<extra_id_1>Assistant\nThe monk uses several astrological factors to match Lhos, including the year, month, day, hour, and minute of birth. They also consider the child’s gender, birth order, and various planetary positions. They then compare these factors for both children, and assign a compatibility score.\n<extra_id_1>User\nCould you write a code that matches Lhos of a girl and a boy before they marry\n<extra_id_1>Assistant\nSure, I’d be happy to do that! Please give me the birthdates for the girl and boy, and I’ll calculate their Lhos.\n<extra_id_1>User\nMy birthday is 1988 june 26. . What is my Lho\n<extra_id_1>Assistant\nYour Lho is 5.\n<extra_id_1>User\nCould you write a code that finds lho

In [78]:
chpt = '12318'
chpt_data[chpt]['policy_response']

'user\nHow do monks match Lho of a girl and a boy using tibetan calenders for marriages\n<extra_id_1>Assistant\nMonks use specific astrological formulas to calculate the compatibility of a girl’s Lho and a boy’s Lho.\n<extra_id_1>User\nWhat specific astrological formulas they use\n<extra_id_1>Assistant\nThe monk uses several astrological factors to match Lhos, including the year, month, day, hour, and minute of birth. They also consider the child’s gender, birth order, and various planetary positions. They then compare these factors for both children, and assign a compatibility score.\n<extra_id_1>User\nCould you write a code that matches Lhos of a girl and a boy before they marry\n<extra_id_1>Assistant\nSure, I’d be happy to do that! Please give me the birthdates for the girl and boy, and I’ll calculate their Lhos.\n<extra_id_1>User\nMy birthday is 1988 june 26. . What is my Lho\n<extra_id_1>Assistant\nYour Lho is 5.\n<extra_id_1>User\nCould you write a code that finds lho\n<extra_id_

In [69]:
chpt_data[chpt]['llm_judge_response']


' \n\n### Step 1: Understand the instruction\nThe user asks for 25 online slots tournament titles that are:\n- Funny\n- Use cowboy slang words and expressions\n- Don\'t repeat nouns in the various titles more than once\n\n### Step 2: Evaluate Assistant 1\'s response\nAssistant 1 provides 25 titles. However:\n- Titles 3, 4, and 5 are identical: "The Cowboy\'s Slot!" and "The Cowgirl\'s Slot!" but then repeats "The Cowboy\'s Slot!" again. This is repetition and not creative.\n- From title 6 onwards, it\'s all "The Cowdollar\'s Slot!" repeated 20 times. This is clearly not following the instruction to have varied and funny titles without repeating nouns. The repetition of "Slot" and "Cowdollar" violates the "don\'t repeat nouns" rule.\n- The titles are not creative or funny; they are repetitive and lazy.\n- It fails to meet the requirement of 25 unique, funny cowboy-themed titles without noun repetition.\n\n### Step 3: Evaluate Assistant 2\'s response\nAssistant 2 starts with a few titles

In [ ]:
chpt_data[chpt]['llm_judge_verdict']


In [ ]:
df_filtered = df_unique[df_unique[column_max] > df_unique['reference_reward']]
len(df_filtered)

In [ ]:
df_filtered[column_max].mean(), df_filtered[column_min].mean()

In [ ]:
uncertainty_reference = np.abs(df_filtered['reference_reward_1'] - df_filtered['reference_reward_2'])
mu_U = uncertainty_reference.mean()
sigma_U = uncertainty_reference.std(ddof=1)

mu_U, sigma_U

In [ ]:
uncertainty_adv = np.abs(df_filtered[column_max] - df_filtered[column_min])
z_score = (uncertainty_adv - mu_U) / sigma_U

In [ ]:
df_tested = df_filtered[z_score > 1.96]
df_tested.__len__()

In [ ]:
df_tested['completion'].str.contains('⚙').sum()

In [ ]:
df_unique_prompt = df_tested.drop_duplicates(subset=['prompt'], keep='first')
df_unique_prompt.__len__()

In [ ]:
df_unique_prompt[-10:]

In [ ]:
with open('data/train.json', 'r') as f:
    train = json.load(f)

In [ ]:
list(train[1000].keys())

In [ ]:
df_unique_prompt.loc[:, 'best_reference'] = np.where(
    df_unique_prompt['reference_reward_1'] > df_unique_prompt['reference_reward_2'],
    df_unique_prompt['reference_response_1'],
    df_unique_prompt['reference_response_2']
)

df_unique_prompt.loc[:, 'chosen_reward'] = np.where(
    df_unique_prompt['reference_reward_1'] > df_unique_prompt['reference_reward_2'],
    df_unique_prompt['reference_reward_1'],
    df_unique_prompt['reference_reward_2']
)

df_unique_prompt.loc[:, 'rejected_reward'] = df_unique_prompt[column_max]

In [ ]:
train[1000]

In [ ]:
df_unique_prompt.loc[:, 'chosen'] = df_unique_prompt['prompt'] + df_unique_prompt['best_reference']

df_unique_prompt.loc[:, 'rejected'] = df_unique_prompt['prompt'] + df_unique_prompt['completion']

df_unique_prompt.loc[:, 'does_gold_agree_with_original'] = True
df_unique_prompt.loc[:, 'preference_strength'] = df_unique_prompt[column_max] - df_unique_prompt[column_min]
df_cleaned = df_unique_prompt.drop([
    'prompt',
    'completion',
    'best_reference',
    column_max,
    column_min,
    # 'completion_ids',
], axis=1)

df_cleaned

In [ ]:
df_cleaned.columns.tolist()

In [ ]:
# make dir
import os
if not os.path.exists('out'):
    os.makedirs('out')
# save the cleaned DataFrame to a JSON file
df_cleaned.to_json('out/train.json', orient='records', lines=False, force_ascii=False)

In [ ]:
# read the saved JSON file to verify
with open('out/train.json', 'r') as f:
    train_data = json.load(f)

train_data[100]